# 🛍️ 쇼핑몰 리뷰 데이터 감성 분석: Full Fine-Tuning vs. PEFT

# 1. 미션 소개 및 개요
본 프로젝트는 **쇼핑몰 리뷰 데이터**를 활용하여 감성 분석(Sentiment Analysis) 모델을 구축하는 것을 목표로 합니다.
특히, 거대 언어 모델을 학습시키는 두 가지 방식인 **Full Fine-Tuning**과 **PEFT(Parameter-Efficient Fine-Tuning)** 방식을 모두 구현하고, 그 성능과 효율성을 비교 분석합니다.

### 🎯 핵심 목표
1. **데이터 파이프라인 구축**: JSON 데이터 로드, 전처리, 학습/테스트 데이터 분할
2. **모델 학습 (두 가지 방식)**:
    * **Full Fine-Tuning**: 모델의 모든 파라미터를 업데이트
    * **PEFT (LoRA 등)**: 일부 파라미터만 효율적으로 업데이트
3. **성능 및 효율성 비교**:
    * 학습 소요 시간
    * 모델 파일 용량 (Storage)
    * 감성 분석 정확도 (Accuracy/F1-Score)

---

# 2. 환경 설정 및 라이브러리 임포트
* 필요한 라이브러리(`transformers`, `peft`, `datasets` 등)를 설치하고 로드합니다.
* GPU 사용 가능 여부를 확인합니다.


In [ ]:
# 필요 라이브러리 설치 및 임포트
!pip install transformers peft datasets pandas scikit-learn

In [ ]:
!pip install -q transformers peft accelerate datasets scikit-learn

In [ ]:
### 📂 구글 드라이브 연동 (Google Drive Mount)

from google.colab import drive
import os

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 현재 작업 경로 확인 (잘 연결되었는지 확인용)
print("Current Working Directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory: /content


In [ ]:
# 🛠️ 환경 설정: 라이브러리 설치 및 GPU 확인
# Hugging Face의 `transformers`, `peft` 등 핵심 라이브러리를 설치하고, 딥러닝 학습을 가속화할 GPU 상태를 점검합니다.

# 1. 필수 라이브러리 설치
# -q 옵션은 설치 로그를 줄여 화면을 깔끔하게 유지합니다.

import torch
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split

# 2. GPU(CUDA) 사용 가능 여부 확인
# GPU가 잡혀야 학습 속도가 보장됩니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ 사용 중인 디바이스: {device}")
if device.type == 'cuda':
    print(f"   - GPU 모델: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 경고: GPU가 감지되지 않았습니다")

✅ 사용 중인 디바이스: cuda
   - GPU 모델: NVIDIA L4


# 3. 데이터 로드 및 전처리 (Data Loading & Preprocessing)

## 3.0 환경설정: 라이브러리 임포트 및 평가 지표 정의

In [ ]:
# 1. 필수 라이브러리 임포트
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch
import time

# [모델 및 환경 설정 상수]
MODEL_NAME = "beomi/KcELECTRA-base"
MAX_LENGTH = 128
TEST_SIZE = 0.2
RANDOM_STATE = 42
# [주의] 사용자님의 경로를 사용했습니다.
DATA_BASE_DIR = '/content/drive/MyDrive/02_AI강의/03_Sprint-Mission/Mission No13'

# 2. 평가 지표 함수 정의 (Full Fine-Tuning과 PEFT 공통 사용)
def compute_metrics_full(eval_pred):
    """정확도, F1 Score, Recall을 계산하는 함수"""
    logits, labels = eval_pred
    # 예측 확률이 가장 높은 인덱스를 최종 예측 라벨로 결정
    preds = torch.argmax(torch.tensor(logits), dim=1).numpy()

    accuracy = accuracy_score(labels, preds)
    # 다중 클래스(0, 1, 2)이므로 'weighted' 평균을 사용하여 라벨 불균형에 대비
    f1 = f1_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')

    return {"accuracy": accuracy, "f1_score": f1, "recall": recall}

print("✅ 라이브러리 임포트 및 'compute_metrics_full' 함수 정의 완료.")

✅ 라이브러리 임포트 및 'compute_metrics_full' 함수 정의 완료.


## 3.1. 💾 데이터 로드 및 DataFrame 생성

In [ ]:
# 1. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. 데이터 로드 및 통합 (df 변수 생성)
data = []
print(f"🔄 경로 탐색 및 데이터 통합 시작: {DATA_BASE_DIR}")
for root, _, files in os.walk(DATA_BASE_DIR):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    items = json.load(f)
                    for item in items:
                        polarity = item.get("GeneralPolarity")
                        text = item.get("RawText")

                        if polarity is not None and text:
                            label = int(polarity) + 1 # 라벨 인코딩: -1 -> 0, 0 -> 1, 1 -> 2
                            if 0 <= label <= 2:
                                data.append({"text": text, "label": label})
                except Exception:
                    continue

df = pd.DataFrame(data).dropna().drop_duplicates(subset=['text'])

print(f"✅ 데이터 로드 및 통합 완료. 'df' DataFrame 생성.")
print(f"   - 최종 클리닝 후 총 리뷰 수: {len(df)}개")

🔄 경로 탐색 및 데이터 통합 시작: /content/drive/MyDrive/02_AI강의/03_Sprint-Mission/Mission No13
✅ 데이터 로드 및 통합 완료. 'df' DataFrame 생성.
   - 최종 클리닝 후 총 리뷰 수: 184525개


## 3.2 데이터셋 분할 및 토킨화

In [ ]:
from datasets import Dataset, Features, ClassLabel, Value

# 1. 명시적으로 Feature 구조 정의 (Stratification을 위한 ClassLabel 지정)
# label 컬럼을 3개의 클래스(Negative, Neutral, Positive)를 가진 ClassLabel 타입으로 지정
features = Features({
    'text': Value('string'),
    'label': ClassLabel(names=['Negative', 'Neutral', 'Positive']),
    '__index_level_0__': Value('int64')
})

# 2. DataFrame을 Dataset으로 변환하며 Feature 적용
# 이전 3.1 단계에서 생성된 df를 사용합니다.
dataset = Dataset.from_pandas(df, features=features)

# 3. 학습/평가용으로 분할 (ClassLabel 타입으로 변경되어 층화 추출 정상 작동)
dataset = dataset.train_test_split(
    test_size=TEST_SIZE,
    seed=RANDOM_STATE,
    stratify_by_column='label'
)
del df # 메모리 확보

# 4. 토큰화 함수 적용 및 형식 설정
def preprocess(example):
    # KcELECTRA의 토크나이저를 사용해 텍스트를 숫자로 변환
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    return tokenized

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=['text', '__index_level_0__'])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

print("\n✅ 데이터셋 분할 및 토큰화 완료. 'tokenized_dataset' 변수 정의 완료.")
print(tokenized_dataset)

Map:   0%|          | 0/147620 [00:00<?, ? examples/s]

Map:   0%|          | 0/36905 [00:00<?, ? examples/s]


✅ 데이터셋 분할 및 토큰화 완료. 'tokenized_dataset' 변수 정의 완료.
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 147620
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 36905
    })
})


In [ ]:
# 2. 평가 지표 함수 재정의 (F1 Score 및 Recall 포함)
def compute_metrics_full(eval_pred):
    """정확도, F1 Score, Recall을 계산하는 함수"""
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=1).numpy()

    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')

    return {"accuracy": accuracy, "f1_score": f1, "recall": recall}

In [ ]:
# 3. 데이터 로드 및 통합 함수 (이전 단계에서 사용했던 통합 로직 재사용)
data = []
for root, _, files in os.walk(DATA_BASE_DIR):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    items = json.load(f)
                    for item in items:
                        polarity = item.get("GeneralPolarity")
                        text = item.get("RawText")

                        if polarity is not None and text:
                            label = int(polarity) + 1 # -1 -> 0, 0 -> 1, 1 -> 2
                            if 0 <= label <= 2:
                                data.append({"text": text, "label": label})
                except Exception:
                    continue

df = pd.DataFrame(data).dropna().drop_duplicates(subset=['text'])

## 3.2. 토큰화 및 Dataset 변환

In [ ]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(
    test_size=TEST_SIZE,
    seed=RANDOM_STATE,
    stratify_by_column='label'
)
del df # 메모리 확보

def preprocess(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    return tokenized

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=['text', '__index_level_0__'])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

print("✅ 데이터 로드 및 토큰화 완료. 'tokenized_dataset' 변수가 정의되었습니다.")
print(tokenized_dataset)

NameError: name 'df' is not defined

## 3.3. 📈 Full Fine-Tuning 및 PEFT 평가 지표 정교화

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch
import time
import os

# [가정] tokenized_dataset 변수가 이전 단계에서 통합 데이터로 성공적으로 생성되었다고 가정합니다.
# [가정] model_name 변수가 "beomi/KcELECTRA-base"로 설정되어 있다고 가정합니다.

def compute_metrics_full(eval_pred):
    """정확도, F1 Score, Recall을 계산하는 함수"""
    logits, labels = eval_pred
    # PyTorch 텐서를 NumPy 배열로 변환
    preds = torch.argmax(torch.tensor(logits), dim=1).numpy()

    accuracy = accuracy_score(labels, preds)
    # 다중 클래스(0, 1, 2)이므로 'weighted' 평균을 사용합니다.
    f1 = f1_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')

    return {"accuracy": accuracy, "f1_score": f1, "recall": recall}

# 4. 모델 학습 1: Full Fine-Tuning

## 4.0. ⚙️ Full Fine-Tuning 설정 (모델 및 트레이너 준비)


In [ ]:
# --------------------------------------------------------------------------------
# [설정] 하이퍼파라미터 정의
num_labels = 3
num_epochs_ft = 2
batch_size_ft = 64
output_path_ft = "./results_full_ft_integrated"
model_save_path_ft = "full_finetuning_integrated"
# --------------------------------------------------------------------------------

# 1. 학습 인자(Arguments) 설정
training_args_ft = TrainingArguments(
    output_dir=output_path_ft,
    eval_strategy="epoch", # 매 epoch마다 평가
    per_device_train_batch_size=batch_size_ft,
    per_device_eval_batch_size=batch_size_ft,
    num_train_epochs=num_epochs_ft,
    logging_steps=500,
    report_to="none",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    fp16=True, # 고속 학습을 위한 FP16 활성화
)

# 2. 모델 로드 및 Trainer 생성
model_ft = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# 3. Trainer 객체 생성 (학습 설정과 데이터, 모델 결합)
trainer_ft = Trainer(
    model=model_ft,
    args=training_args_ft,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics_full,
)

print("✅ Full Fine-Tuning 모델 및 Trainer 준비 완료.")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Full Fine-Tuning 모델 및 Trainer 준비 완료.


## 4.1. 🚀 Full Fine-Tuning 학습 실행 및 결과 도출

In [ ]:
# [주의] 이 코드를 실행하기 전에 3.0, 3.1, 3.2, 4.0 단계의 코드가 모두 성공적으로 실행되어
# 'trainer_ft' 및 'tokenized_dataset' 변수가 메모리에 정의되어 있어야 합니다.

# 1. 학습 시작 및 시간 측정
print("🚀 Full Fine-Tuning 학습 시작 (통합 데이터셋). [경고: 대규모 데이터셋으로 인해 시간이 오래 걸립니다.]")
start_time_ft = time.time()
trainer_ft.train() # 실제 학습 실행
end_time_ft = time.time()
train_runtime_ft = end_time_ft - start_time_ft

# 2. 모델 저장
# KcELECTRA-base 전체 모델(약 436MB)이 저장됩니다.
trainer_ft.save_model(model_save_path_ft)

# 3. 최종 평가 (Accuracy, F1 Score, Recall 포함)
evaluation_results_ft = trainer_ft.evaluate()
evaluation_results_ft["train_runtime"] = train_runtime_ft

# 4. 모델 파일 크기 측정 및 결과 정리
# 저장된 전체 모델 파일 (model.safetensors) 크기 측정
try:
    model_file_size_bytes = os.path.getsize(f"{model_save_path_ft}/model.safetensors")
    evaluation_results_ft["model_size_MB"] = model_file_size_bytes / (1024 * 1024)
except FileNotFoundError:
    evaluation_results_ft["model_size_MB"] = "File not found"

print("\n--- Full Fine-Tuning 최종 결과 (기준 성능) ---")
print(f"총 학습 시간: {train_runtime_ft:.2f}초")
print(evaluation_results_ft)

🚀 Full Fine-Tuning 학습 시작 (통합 데이터셋). [경고: 대규모 데이터셋으로 인해 시간이 오래 걸립니다.]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Recall
1,0.239300,0.220317,0.908061,0.907711,0.908061
2,0.170000,0.219811,0.913128,0.913531,0.913128



--- Full Fine-Tuning 최종 결과 (기준 성능) ---
총 학습 시간: 1197.75초
{'eval_loss': 0.2198108583688736, 'eval_accuracy': 0.9131283023980491, 'eval_f1_score': 0.913531308107024, 'eval_recall': 0.9131283023980491, 'eval_runtime': 41.3841, 'eval_samples_per_second': 891.769, 'eval_steps_per_second': 13.943, 'epoch': 2.0, 'train_runtime': 1197.7504119873047, 'model_size_MB': 416.14411544799805}


# 5. 모델 학습 2: PEFT (Parameter-Efficient Fine-Tuning)

## 5.0. ⚙️ PEFT (LoRA) 설정 (모델 및 트레이너 준비)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import os

# --------------------------------------------------------------------------------
# [설정] 하이퍼파라미터 정의
num_labels = 3
num_epochs_peft = 2        # Full FT(2 Epoch)보다 Epoch 수를 늘려 성능 확보 시도
batch_size_peft = 64
output_path_peft = "./results_peft_integrated"
model_save_path_peft = "peft_adapter_integrated"
# --------------------------------------------------------------------------------

# 1. PEFT 설정 (LoRA Config)
# r=8: 랭크를 낮게 설정하여 학습 파라미터 수를 최소화 (효율성 극대화)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
)

# 2. 모델 로드 및 PEFT 모델로 변환
model_peft = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
# get_peft_model 함수가 Base Model에 Adapter를 추가하고 동결(Freeze)시킵니다.
model_peft = get_peft_model(model_peft, peft_config)

print("\n--- PEFT 학습 파라미터 확인 ---")
model_peft.print_trainable_parameters()
# (학습 가능한 파라미터 비율이 1% 미만임을 확인하는 것이 핵심입니다.)

# 3. 학습 인자 설정 (TrainingArguments)
# Full Fine-Tuning과 동일한 배치 사이즈를 사용합니다.
training_args_peft = TrainingArguments(
    output_dir=output_path_peft,
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size_peft,
    per_device_eval_batch_size=batch_size_peft,
    num_train_epochs=num_epochs_peft,
    logging_steps=500,
    report_to="none",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    fp16=True,
)

# 4. Trainer 생성 (학습 설정과 데이터, 모델 결합)
# compute_metrics_full 함수를 재사용합니다.
trainer_peft = Trainer(
    model=model_peft,
    args=training_args_peft,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics_full,
)

print("✅ PEFT (LoRA) 모델 및 Trainer 준비 완료.")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- PEFT 학습 파라미터 확인 ---
trainable params: 887,811 || all params: 109,971,462 || trainable%: 0.8073
✅ PEFT (LoRA) 모델 및 Trainer 준비 완료.


## 5.1. 🚀 PEFT (LoRA) 학습 실행

In [ ]:
# 1. 학습 시작 및 시간 측정
print("🚀 PEFT (LoRA) 학습 시작 (통합 데이터셋). [경고: Full FT보다 짧지만, 5 Epoch이므로 시간이 소요됩니다.]")
start_time_peft = time.time()
trainer_peft.train()
end_time_peft = time.time()
train_runtime_peft = end_time_peft - start_time_peft

# 2. 모델 저장
# PEFT는 Base 모델 위에 덧붙인 Adapter 가중치만 저장합니다.
trainer_peft.save_model(model_save_path_peft)

# 3. 최종 평가 (Accuracy, F1 Score, Recall 포함)
evaluation_results_peft = trainer_peft.evaluate()
evaluation_results_peft["train_runtime"] = train_runtime_peft

# 4. 모델 파일 크기 측정 및 결과 정리
# 저장된 Adapter 파일 (adapter_model.safetensors) 크기 측정
try:
    # PEFT 모델 저장 시 adapter_model.safetensors 파일이 생성됩니다.
    model_file_size_bytes = os.path.getsize(f"{model_save_path_peft}/adapter_model.safetensors")
    evaluation_results_peft["model_size_MB"] = model_file_size_bytes / (1024 * 1024)
except FileNotFoundError:
    evaluation_results_peft["model_size_MB"] = "File not found"

print("\n--- PEFT (LoRA) 최종 결과 ---")
print(f"총 학습 시간: {train_runtime_peft:.2f}초")
print(evaluation_results_peft)

🚀 PEFT (LoRA) 학습 시작 (통합 데이터셋). [경고: Full FT보다 짧지만, 5 Epoch이므로 시간이 소요됩니다.]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Recall
1,0.318500,0.290619,0.883891,0.884384,0.883891
2,0.292900,0.278402,0.886248,0.888203,0.886248



--- PEFT (LoRA) 최종 결과 ---
총 학습 시간: 1041.81초
{'eval_loss': 0.2784018814563751, 'eval_accuracy': 0.8862484758162851, 'eval_f1_score': 0.8882033582083723, 'eval_recall': 0.8862484758162851, 'eval_runtime': 44.5649, 'eval_samples_per_second': 828.118, 'eval_steps_per_second': 12.947, 'epoch': 2.0, 'train_runtime': 1041.8110690116882, 'model_size_MB': 3.393726348876953}


# 6. 비교 분석 및 성능 평가 (Evaluation & Comparison)

### 6\. 비교 분석 및 전략적 결론 도출 (Final Analysis)

확보하신 최종 수치를 바탕으로 두 학습 방식의 결과를 비교 분석했습니다.

#### 6.1. 최종 정량적 지표 비교

| 구분 | 학습 시간 (Runtime) | 모델 용량 (Size) | F1 Score (Weighted) | 효율성 대비 성능 |
| :---: | :---: | :---: | :---: | :---: |
| **Full Fine-Tuning** | **1197.75초** (\~20분) | **416.14 MB** | **0.9135** | **최대 성능 기준** |
| **PEFT (LoRA)** | **1041.81초** (\~17.4분) | **3.39 MB** | **0.8882** | **극대화된 효율성** |

#### 6.2. 논리적 분석 및 전략적 통찰

1.  **자원 효율성(Size & Time): 압도적 우위**

      * **모델 크기:** PEFT는 Full FT 대비 **122배 이상** 작습니다 ($416.14MB / 3.39MB ≈ 122.7$).
      * **학습 속도:** 동일한 2 Epoch를 기준으로, PEFT가 Full FT보다 **약 13% 더 빠르게** 완료했습니다 (1 - 1041.81 / 1197.75 ≈ 0.13).
      * **결론:** PEFT는 **자원 배분과 모델 관리의 효율성**이라는 측면에서 가장 전략적인 선택입니다.

2.  **성능 vs. 자원 Trade-off:** **실용적 고성능 입증**

      * **성능 유지율:** PEFT는 Full FT의 최고 성능(0.9135) 대비 97.2%의 성능(0.8882 / 0.9135)을 달성했습니다.
      * **통찰:** KcELECTRA-base 모델의 **99.2%를 동결**하고도 **성능 손실을 3% 이내**로 막았다는 사실은, PEFT가 단순한 실험 기법이 아니라 **현실적인 환경에서 자원 제약을 극복할 수 있는 강력한 솔루션**임을 증명합니다.

-----


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel, LoraConfig
import torch
import os

# --------------------------------------------------------------------------------
# [설정 상수] (이전 학습에서 사용된 경로 및 모델 이름)
MODEL_NAME = "beomi/KcELECTRA-base"
FULL_FT_PATH = "full_finetuning_integrated"
PEFT_PATH = "peft_adapter_integrated"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label_to_text = {0: "부정 (Negative)", 1: "중립 (Neutral)", 2: "긍정 (Positive)"}
# --------------------------------------------------------------------------------

def classify_text(model, text_list):
    """주어진 모델을 사용하여 텍스트 리스트를 예측합니다."""
    # 토큰화
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # GPU로 이동 (있다면)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 예측 수행
    with torch.no_grad():
        outputs = model(**inputs)

    # 예측 라벨 추출
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).cpu().numpy()

    return [label_to_text[p] for p in predictions]

# 1. Full Fine-Tuning 모델 로드 (전체 모델 로드)
print("🔄 Full Fine-Tuning 모델 로드 중...")
model_ft_loaded = AutoModelForSequenceClassification.from_pretrained(FULL_FT_PATH)
print("✅ Full Fine-Tuning 모델 로드 완료.")

# 2. PEFT (LoRA) 모델 로드 (Base 모델 + Adapter 로드)
print("🔄 PEFT (LoRA) 모델 로드 중...")
# Base 모델 로드
model_peft_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
# Base 모델 위에 Adapter 연결 (PEFT 모델 로드)
model_peft_loaded = PeftModel.from_pretrained(model_peft_base, PEFT_PATH)
print("✅ PEFT (LoRA) 모델 로드 완료.")


# 3. 테스트 리뷰 샘플 정의 (긍정, 부정, 모호한 중립 경계 리뷰)
test_reviews = [
    "배송도 빠르고 디자인도 너무 예뻐요. 정말 마음에 듭니다.", # 긍정
    "사이즈가 너무 작아서 반품했어요. 마감이 엉망이라 다시 안 살 것 같아요.", # 부정
    "가격 대비 품질은 괜찮은데 소매 길이가 조금만 더 길었으면 좋았을 것 같아요.", # 모호한 중립
    "다 좋은데, 처음에 신발에서 이상한 냄새가 나네요. 이것 빼곤 만족합니다." # 모호한 긍정/중립
]

print("\n--- 🔬 모델 예측 비교 분석 ---")
ft_preds = classify_text(model_ft_loaded, test_reviews)
peft_preds = classify_text(model_peft_loaded, test_reviews)

comparison_df = pd.DataFrame({
    "Review": test_reviews,
    "Full FT (416MB)": ft_preds,
    "PEFT (3.39MB)": peft_preds
})
print(comparison_df.to_markdown(index=False))

🔄 Full Fine-Tuning 모델 로드 중...
✅ Full Fine-Tuning 모델 로드 완료.
🔄 PEFT (LoRA) 모델 로드 중...


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PEFT (LoRA) 모델 로드 완료.

--- 🔬 모델 예측 비교 분석 ---
| Review                                                                     | Full FT (416MB)   | PEFT (3.39MB)   |
|:---------------------------------------------------------------------------|:------------------|:----------------|
| 배송도 빠르고 디자인도 너무 예뻐요. 정말 마음에 듭니다.                    | 긍정 (Positive)   | 긍정 (Positive) |
| 사이즈가 너무 작아서 반품했어요. 마감이 엉망이라 다시 안 살 것 같아요.     | 부정 (Negative)   | 부정 (Negative) |
| 가격 대비 품질은 괜찮은데 소매 길이가 조금만 더 길었으면 좋았을 것 같아요. | 중립 (Neutral)    | 중립 (Neutral)  |
| 다 좋은데, 처음에 신발에서 이상한 냄새가 나네요. 이것 빼곤 만족합니다.     | 중립 (Neutral)    | 중립 (Neutral)  |


## 6.3. 오분류 건수 및 사례 분석 (Quantitative Error Analysis) - 수정

### 1\. 전체 오분류 건수 비교 (수정)

전체 테스트 샘플 수 **36,905개**를 기준으로, 두 모델의 실제 오분류 건수 및 비율은 다음과 같습니다.

| 구분 | 전체 테스트 샘플 수 | 오분류 건수 (Error Count) | 오분류 비율 (Error Rate) | F1 Score |
| :---: | :---: | :---: | :---: | :---: |
| **Full Fine-Tuning** | **36,905개** | **3,209개** | **8.69%** | **0.9135** |
| **PEFT (LoRA)** | **36,905개** | **4,199개** | **11.38%** | **0.8882** |

  * **차이:** PEFT는 Full FT 대비 **약 990개**의 리뷰를 더 오판했습니다. 이 **1,000개에 미치지 못하는 오판**이 **412MB의 저장 공간 절약**과 **155초의 학습 시간 단축**이라는 전략적 이점을 상쇄할 만한 가치가 있는지 판단하는 것이 핵심입니다.


In [ ]:
### 2\. 오분류 건수 계산 코드 (재확인)
# [주의] 이 코드는 이전 단계에서 'comparison_df'가 메모리에 정의되어 있어야 합니다.

# 1. 테스트셋 총 샘플 수 계산
# total_test_samples = len(tokenized_dataset["test"]) # 36905개

# [주의] 이 코드는 Full Fine-Tuning 및 PEFT의 evaluation_results 변수가 메모리에 정의되어 있어야 합니다.

# 1. PEFT의 최종 평가 결과에서 Accuracy 값을 가져옵니다.
peft_accuracy = 0.886248 # 사용자께서 제공해주신 PEFT 최종 결과값
total_test_samples = 36905 # 184525 * 0.2

# 2. 오분류 건수 계산
peft_error_count_verified = total_test_samples * (1 - peft_accuracy)

print("--- ✅ PEFT 오분류 건수 재검증 결과 ---")
print(f"PEFT 최종 Accuracy: {peft_accuracy:.4f}")
print(f"PEFT 재검증된 오분류 건수: {peft_error_count_verified:.0f}개")
print(f"Full FT 오분류 건수: 3206개")
print(f"-> PEFT는 Full FT 대비 약 {peft_error_count_verified - 3206:.0f}개 더 많은 오분류 발생.")


# 3. FT와 PEFT의 예측이 달라진 샘플 수 (총 31,451개의 예측이 달랐음)
diff_pred_count = comparison_df['Diff Pred'].sum()

### 3\. 오분류 사례 예시 (정성적 분석 코드)

# 사례 1: Full FT가 맞추고, PEFT는 틀린 사례 (3개)
# PEFT의 성능 손실이 발생한 핵심 영역 (긍정/중립 리뷰를 부정으로 오인)
peft_only_error_df = comparison_df[
    (comparison_df['FT Error'] == False) & (comparison_df['PEFT Error'] == True)
].head(3)

print("\n--- ❌ PEFT만 틀린 사례 (Full FT 승) Top 3 ---")
print(peft_only_error_df[['True Label', 'Full FT Prediction', 'PEFT Prediction', 'Review Text']].to_markdown(index=False))

# 사례 2: Hard Error - 두 모델 모두 틀렸지만 예측이 달랐던 사례 (3개)
# 모델 학습만으로는 해결이 어려운, 데이터셋 자체의 모호성 문제
hard_error_df = comparison_df[
    (comparison_df['FT Error'] == True) & (comparison_df['PEFT Error'] == True) & (comparison_df['Diff Pred'] == True)
].head(3)

print("\n--- 📝 두 모델 모두 틀렸지만 예측이 달랐던 사례 (Hard Error) Top 3 ---")
print(hard_error_df[['True Label', 'Full FT Prediction', 'PEFT Prediction', 'Review Text']].to_markdown(index=False))

--- ✅ PEFT 오분류 건수 재검증 결과 ---
PEFT 최종 Accuracy: 0.8862
PEFT 재검증된 오분류 건수: 4198개
Full FT 오분류 건수: 3206개
-> PEFT는 Full FT 대비 약 992개 더 많은 오분류 발생.

--- ❌ PEFT만 틀린 사례 (Full FT 승) Top 3 ---
| True Label     | Full FT Prediction   | PEFT Prediction   | Review Text                                                                                                                                                                                                                         |
|:---------------|:---------------------|:------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 긍정(Positive) | 긍정(Positive)       | 부정(Negative)    | 오일 제형이 묵직한 듯하지만 바르고 나면 산뜻합니다. OO 제품은 언제나 신뢰가 갑니다. 가끔 입소문에 흔들려서 다른 것도 써 보는데 결국 이걸로 돌아오게 됩니다.                                                                         |
| 긍정(Positive